In [45]:
import os
import json
from dotenv import load_dotenv
from groq import Groq
from tavily import TavilyClient

In [46]:
load_dotenv()

True

In [47]:
client = Groq()
tavily = TavilyClient()

In [48]:
# web search 

sources = []      # every url the searches returned
queries = []      # every query the LLM decided to search

def web_search(query: str) -> str:
    queries.append(query)
    results = tavily.search(query, max_results=5)["results"]
    
    for r in results:
        sources.append({"title": r["title"], "url": r["url"]})
    return json.dumps([{"title": r["title"], "url": r["url"], "content": r["content"]} for r in results])

In [49]:
tools = [{
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Search the web for current information. Use when the question needs recent or factual data.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "search query"}},
            "required": ["query"],
        },
    },
}]

In [50]:
# get user inputs here or add prompts
PROMPTS = []
prompt = "ipl 2026 winner"
messages = [{"role": "user", "content": prompt}]

In [51]:
MODEL, TEMP = "openai/gpt-oss-120b", 0.0

for _ in range(3):  # max 3 rounds so it can't loop forever
    response = client.chat.completions.create(
        model=MODEL, temperature=TEMP, messages=messages, tools=tools, tool_choice="auto",
    )
    msg = response.choices[0].message

    if not msg.tool_calls:   # LLM answered without (more) searching
        break

    messages.append({
        "role": "assistant",
        "content": msg.content or "",
        "tool_calls": [
            {"id": tc.id, "type": "function",
             "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ],
    })
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": web_search(**args)})

In [52]:
print(f"Answer:\n{msg.content}")
print(f"\nQueries: {queries}")
print(f"Sources:\n{json.dumps(sources, indent=2)}")

Answer:
The winner of the 2026 Indian Premier League (IPL) was **Royal Challengers Bengaluru (RCB)**, who defeated the Gujarat Titans in the final to claim the title.

Queries: ['IPL 2026 winner']
Sources:
[
  {
    "title": "Home",
    "url": "https://www.iplt20.com"
  },
  {
    "title": "Cricket Live Score, Schedule, Points Table & Team Stats - IPL.com",
    "url": "https://www.ipl.com"
  },
  {
    "title": "Indian Premier League - Wikipedia",
    "url": "https://en.wikipedia.org/wiki/Indian_Premier_League"
  },
  {
    "title": "2026 Indian Premier League - Wikipedia",
    "url": "https://en.wikipedia.org/wiki/2026_Indian_Premier_League"
  },
  {
    "title": "Live Cricket Score, Schedule & Results | IPL.com",
    "url": "https://www.ipl.com/matches"
  }
]
